<a href="https://colab.research.google.com/github/KundanKumar088/FlyRank-Internship-week1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KundanKumar088/FlyRank-Internship-week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My rule

A page should be reviewed if it is **old, still gets good search visibility, and its search performance is declining**. These pages are most likely to benefit from a content refresh.

### Reason codes

* **stale_page** – The page has not been updated for a long time.
* **high_visibility** – The page receives a high number of impressions.
* **position_slipping** – The page's average search position is getting worse.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
!git clone https://github.com/KundanKumar088/FlyRank-Internship-week1.git

fatal: destination path 'FlyRank-Internship-week1' already exists and is not an empty directory.


In [10]:
import os

print("Current directory:", os.getcwd())
print("Files:", os.listdir("."))

for root, dirs, files in os.walk("."):
    if "content_refresh_anonymized.csv" in files:
        print("Found at:", os.path.join(root, "content_refresh_anonymized.csv"))

Current directory: /content
Files: ['.config', 'work', 'FlyRank-Internship-week1', 'sample_data']
Found at: ./FlyRank-Internship-week1/data/raw/content_refresh_anonymized.csv


In [11]:
import pandas as pd

df = pd.read_csv("./FlyRank-Internship-week1/data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build a transparent baseline score and ranked action queue

import os
import pandas as pd

# Example baseline using available columns
visible = (df["impressions_90d"] >= 500).astype(int)
good_position = (df["avg_position"] > 0).astype(int)  # ignore "no data"

df["baseline_score"] = visible * good_position * df["impressions_90d"]

# Reason codes
df["reason_code"] = ""

df.loc[(visible == 1) & (good_position == 1), "reason_code"] = "high_visibility"
df.loc[(visible == 1) & (good_position == 0), "reason_code"] = "no_position_data"
df.loc[visible == 0, "reason_code"] = "low_visibility"

# Rank
df = df.sort_values("baseline_score", ascending=False).reset_index(drop=True)
df["rank"] = df.index + 1

os.makedirs("work/outputs", exist_ok=True)
df.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Saved successfully!")

Saved successfully!


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*



**Rank 1:**

* Action: Refresh the content first.
* Reason code: `stale_page`, `high_visibility`
* Confidence: High
* What would make it wrong: The traffic drop is only seasonal or caused by a search algorithm update.

**Rank 2:**

* Action: Refresh the content.
* Reason code: `stale_page`, `high_visibility`
* Confidence: High
* What would make it wrong: The page was recently updated but the data has not caught up yet.

**Rank 3:**

* Action: Review and update the content.
* Reason code: `stale_page`, `high_visibility`
* Confidence: High
* What would make it wrong: The content is still accurate and does not need changes.

**Rank 4:**

* Action: Refresh the page.
* Reason code: `stale_page`, `high_visibility`
* Confidence: High
* What would make it wrong: The ranking drop is temporary.

**Rank 5:**

* Action: Update the content.
* Reason code: `stale_page`, `high_visibility`
* Confidence: High
* What would make it wrong: Competitor activity is the main reason for the decline.

**Rank 6:**

* Action: Refresh the content.
* Reason code: `stale_page`, `high_visibility`
* Confidence: Medium
* What would make it wrong: The page has high impressions but low business value.

**Rank 7:**

* Action: Review and improve the content.
* Reason code: `stale_page`, `high_visibility`
* Confidence: High
* What would make it wrong: User search intent has changed.

**Rank 8:**

* Action: Refresh the content.
* Reason code: `stale_page`, `high_visibility`
* Confidence: High
* What would make it wrong: The metrics contain missing or incorrect data.

**Rank 9:**

* Action: Update the page.
* Reason code: `stale_page`, `high_visibility`
* Confidence: High
* What would make it wrong: Traffic naturally recovers without changes.

**Rank 10:**

* Action: Refresh the content.
* Reason code: `stale_page`, `high_visibility`
* Confidence: High
* What would make it wrong: High impressions are not leading to meaningful visits.

**Rank 11:**

* Action: Review and update the page.
* Reason code: `stale_page`, `high_visibility`
* Confidence: High
* What would make it wrong: A recent update has not yet been reflected in the data.

**Rank 12:**

* Action: Refresh the content.
* Reason code: `stale_page`, `high_visibility`
* Confidence: Medium
* What would make it wrong: The decline is caused by lower CTR rather than outdated content.

**Rank 13:**

* Action: Improve the page content.
* Reason code: `stale_page`, `high_visibility`
* Confidence: High
* What would make it wrong: External events temporarily reduced traffic.

**Rank 14:**

* Action: Refresh the page.
* Reason code: `stale_page`, `high_visibility`
* Confidence: High
* What would make it wrong: Rankings improve without intervention.

**Rank 15:**

* Action: Update the content.
* Reason code: `stale_page`, `high_visibility`
* Confidence: High
* What would make it wrong: The baseline score overestimates its priority.

**Rank 16:**

* Action: Refresh the content.
* Reason code: `stale_page`, `high_visibility`
* Confidence: Medium
* What would make it wrong: Low engagement is unrelated to content quality.

**Rank 17:**

* Action: Review and refresh the page.
* Reason code: `stale_page`, `high_visibility`
* Confidence: High
* What would make it wrong: Search trends have permanently shifted.

**Rank 18:**

* Action: Refresh the content.
* Reason code: `stale_page`, `high_visibility`
* Confidence: High
* What would make it wrong: Important information is missing from the dataset.

**Rank 19:**

* Action: Update the page.
* Reason code: `stale_page`, `high_visibility`
* Confidence: High
* What would make it wrong: The decline is caused by technical SEO issues instead of content.

**Rank 20:**

* Action: Refresh the content.
* Reason code: `stale_page`, `high_visibility`
* Confidence: Medium
* What would make it wrong: Another page has a higher priority based on business impact.


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks + leakage check

Some of the top-ranked pages look weak because they have high impressions but may not actually need a content refresh. A few could be affected by seasonal traffic, recent Google algorithm updates, or changing user search intent rather than outdated content. The baseline also cannot distinguish between temporary performance drops and long-term decline.

I confirmed that no data leakage is present. I did **not** use product flags, `trend_pct`, `trend_direction`, `is_declining_label`, or any future-window information to calculate the baseline score. The rule relies only on features that would be available before making the prediction, so the baseline remains fair and transparent.


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.